In [ ]:
# Setup. Everything below runs the CLI itself, so the notebook and the PBS job
# execute identical code.
import logging
import os
import sys

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, Markdown, display  # noqa: E402

from scripts.run_corrca_raw_weighted import _build_arg_parser, run  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True
)
print("Setup complete.")

# Stage 06b — raw voltages weighted by the CorrCA topographies: PC1 and all data

Per recording: the **raw concatenated voltage** is z-scored per channel and every
electrode is multiplied by its weight (each stored fit's absolute CC 1 and CC 2 shared
topography, the **binary ASSR ROI**, or **uniform** over all electrodes, the stage-00
reference set apart in every figure). The weighted signal is then read two ways:

- **PC1**: the PCA over channels of the weighted trial mean gives PC1, every trial is
  projected onto it, and PC1 is sign-aligned across both conditions;
- **all data**: the mean over all electrodes, no component separated.

Either signal's trials are normalised against the pre-onset window (minus their own
pre-onset mean, divided by the pre-onset SD pooled over trials) and averaged. The
figures compare the weightings with uniform, the conditions with each other, and both
differences. Read-only: the stage-06b stores must already exist, and nothing is
fitted.

**The method is documented in
[`docs/analyses/06b-corrca-condition-comparison/`](../../docs/analyses/06b-corrca-condition-comparison/README.md)**
(section *Raw voltages through the topographies*) — this notebook does not restate it.

**What this notebook is for.** Checking the workflow cheaply before submitting
[`run_raw_weighted.pbs`](../../jobs/metacentrum/06b-corrca-condition-comparison/run_raw_weighted.pbs).
It calls `run()` from
[`scripts/run_corrca_raw_weighted.py`](../../scripts/run_corrca_raw_weighted.py) — the same function the PBS job
calls — at reduced scale, then shows what it wrote.

## Configuration

In [ ]:
# Which stored runs to read. Each full fit and every '_log', '_roi', '_non_roi' fit
# are read, each broadband and per BANDS token; a missing store is skipped with a
# warning. Every run listed here shares one y-scale, as all three do in the job.
EXPERIMENT = "assr"
VARIANTS = [
    "corrca_channel_joined_separate",
    # "corrca_channel_joined_tracks",
    # "corrca_channel_individual_conditions",
]
N_PCA = 5  # the stores' component count (the job's default)
BANDS = ["broadband", "30-50hz"]
COMPONENTS = 2  # weight by CC 1..COMPONENTS (plus the binary ROI and uniform)
SKIP_LOG_POWER = True
SKIP_ROI = False
SKIP_NON_ROI = True

# Fixed y-limits, one for the curves (PC1 and mean alike) and one for the differences (None = the
# largest value drawn in this run). The job picks its own from every run, so pin these
# to compare a notebook figure with a job figure.
YLIM = None
YLIM_DIFF = None

# Subset the cohort and the time axis. There is no channel subset: the weights are
# topographies on the full montage. The z-score and the pre-onset SD come from the
# trimmed recording, so a small N_TIMES is a wiring check, not a result.
N_SUBJECTS = 4  # participants
N_TIMES = 12000  # samples (~48 s at 250 Hz, ~37 trials)

# Participant panels for every fit and weighting add many figures; the job
# draws them all, the notebook shows them only when asked.
SHOW_PARTICIPANTS = False

OUT_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06b-corrca-condition-comparison"
    / "out"
    / "raw_weighted"
)
PLOTS_DIR = OUT_DIR / "plots"

## Run

The same function the PBS job calls. The log lists every store read (or skipped), how
many trials each recording contributed, how well every weighting's PC1 topographies
agree after the sign alignment, and the two shared y-ranges.

In [ ]:
args = _build_arg_parser().parse_args(
    [
        "--experiment",
        EXPERIMENT,
        "--variant",
        *VARIANTS,
        "--n_pca",
        str(N_PCA),
        "--bands",
        *BANDS,
        "--components",
        str(COMPONENTS),
        *(["--skip_log_power"] if SKIP_LOG_POWER else []),
        *(["--skip_roi"] if SKIP_ROI else []),
        *(["--skip_non_roi"] if SKIP_NON_ROI else []),
        *(["--ylim", str(YLIM)] if YLIM else []),
        *(["--ylim_diff", str(YLIM_DIFF)] if YLIM_DIFF else []),
        "--n_subjects",
        str(N_SUBJECTS),
        "--n_times",
        str(N_TIMES),
        "--save_dir",
        str(PLOTS_DIR),
    ]
)
run(args)

## Figures

Per fit and readout (`pc1`, `mean`): the main figure (rows = conditions, their group
means and their paired difference; columns = weightings), `_overlay` (uniform against
each other weighting, per condition) and `_differences` (Psilocybin − Placebo per
weighting against uniform's, then per condition each weighting minus uniform). Both
readouts are in pre-onset SD units and share one y-range for the curves and another for
the differences, across every figure of the run; the stimulus onset is dashed and its
offset dotted. All are drawn by
[`src/visualization/corrca_plots.py`](../../src/visualization/corrca_plots.py), which
the CLI uses too. The weights before the absolute value are the fit's stored
`corrca_shared_patterns`; stage 06b no longer draws them.

In [ ]:
for png in sorted(PLOTS_DIR.rglob("*.png")):
    if "participants" in png.parts and not SHOW_PARTICIPANTS:
        continue
    display(Markdown(f"**{png.relative_to(PLOTS_DIR)}**"))
    display(Image(filename=str(png)))